# Gene-body mCG vs expression (supporting)

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}mcds/Pool_CZDA_PBMC.mcds'`  ·  _mC matrix (mcds)_
- `f'{indir}clustering/merged/5kCG100k3C_summary.h5ad'`  ·  _joint summary obj_
- `f'{indir}clustering/merged/L1pre/merge_5kCG_echo_entex_immune.h5ad'`  ·  _other_
- `'geneCG_expr_corr/PBMC_NaiveMem_gene_mCG.hdf'`  ·  _expression_
- `'PMD_ATAC/gene/bulkexpr_Hao2021.hdf'`  ·  _PMD/methyl-compartment_
- `'PMD_ATAC/gene/design_Hao2021.hdf'`  ·  _PMD/methyl-compartment_
- `f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz'`  ·  _ref: gencode_
- `f'PMD_ATAC/gene/allgene.split50.slop250kb.5kb.{ct}.CGN-Merge.tsv'`  ·  _PMD/methyl-compartment_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import MCDS

import snapatac2 as snap
import warnings
warnings.filterwarnings("ignore")

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
indir = f'{ENTEX_ROOT}/'

In [4]:
mcds = MCDS.open(f'{indir}mcds/Pool_CZDA_PBMC.mcds', var_dim='gene')
mcds

In [5]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs


In [6]:
adata_merge = anndata.read_h5ad(f'{indir}clustering/merged/L1pre/merge_5kCG_echo_entex_immune.h5ad')
adata_merge

In [7]:
selc = meta.index[meta.index.isin(adata_merge.obs.index)]
adata_merge.obs['celltype'] = adata_merge.obs['celltype'].astype(str)
adata_merge.obs[['L1', 'L2_any']] = meta[['L1', 'L2_any']].copy()
# adata_merge.obs.loc[selc, 'celltype'] = meta.loc[selc, 'leiden_cons'].values

In [8]:
ds = 0.5
coord_base = 'tsne'

fig, axes = plt.subplots(4, 3, figsize=(15, 15), dpi=300, constrained_layout=True)

tmp = adata_merge.obs.loc[adata_merge.obs['batch']=='echo']
ax = axes[0,0]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'], c='#e0e0e0', edgecolors='none', s=ds, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype',
                        s=ds*5,
                        labelsize=12,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True,
                        legend_kws={'ncol':1, 'fontsize':6},
                       )
for ax in axes[0, 1:]:
    ax.axis('off')
    
for i,ct in enumerate(['c1', 'c15', 'c7']):
    print(adata_merge.obs.loc[(adata_merge.obs['L1']==ct), 'L2_any'].unique().shape[0])
    for j in range(3):
        tmp = adata_merge.obs.loc[(adata_merge.obs['L1']==ct) & (adata_merge.obs['L2_any'].isin([f'c{k}' for k in np.arange(j*5, (j+1)*5)]))]
        ax = axes[i+1,j]
        ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'], c='#e0e0e0', edgecolors='none', s=ds, rasterized=True)
        _ = categorical_scatter(data=tmp,
                                ax=ax,
                                coord_base=coord_base,
                                hue='L2_any',
                                text_anno='L2_any',
                                s=ds*2,
                                labelsize=12,
                                max_points=None,
                                palette='tab10',
                                scatter_kws={'rasterized':True},
                                show_legend=True)


In [9]:
leg = {'c1':{'Tc-Mem':[1,2,3,5,9,10,11,13,14], 'Th-Mem':[0,4,6,7,8,12]},
       'c15':{'Tc-Naive':[3,4], 'Th-Naive':[0,1,2,5,6,7,8,9,10,11,12]},
       'c7':{'B-Naive':[0,10,11], 'B-Mem':[1,2,3,4,5,6,7,8,9,12]}}


In [10]:
meta_hema = meta.loc[meta['L1'].isin(leg.keys())]
meta_hema['celltype'] = ''
for l1 in leg:
    for ct in leg[l1]:
        selct = [f'c{i}' for i in leg[l1][ct]]
        selc = (meta_hema['L1']==l1) & (meta_hema['L2_any'].isin(selct))
        meta_hema.loc[selc, 'celltype'] = ct
        

In [11]:
meta_hema['cell_group'] = meta_hema['celltype'].astype(str) + '-' + meta_hema['Tissue'].astype(str)

In [12]:
meta_hema[['Tissue', 'celltype']].value_counts().unstack()

In [13]:
meta_hema_pbmc = meta_hema.loc[(meta_hema['Tissue']=='PBMC') & (meta_hema['L1'].isin(leg.keys()))]


In [14]:
selc = mcds.get_index('cell').intersection(meta_hema_pbmc.index)
mcds = mcds.sel({'cell':selc})

In [15]:
mcds = mcds.assign_coords(celltype=('cell', meta_hema_pbmc.loc[mcds.get_index('cell'), 'celltype']))


In [16]:
mcds = mcds.groupby('celltype').sum()
mcds = MCDS(mcds, obs_dim='celltype', var_dim='gene')


In [17]:
mcds = mcds.sel({'mc_type':'CGN'})

In [18]:
cov = mcds['gene_da'].sel(count_type='cov').mean(dim='celltype').squeeze().to_pandas()


In [19]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, log_scale=(True, False), ax=ax)
# ax.set_yscale('log')

In [20]:
mcds = mcds.sel({'gene':cov.index[cov>100]})

black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)

exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [21]:
# mcds.add_mc_frac(normalize_per_cell=False)
mcds['gene_da_frac'] = mcds['gene_da'].sel({'count_type':'mc'}) / mcds['gene_da'].sel({'count_type':'cov'})
data = mcds['gene_da_frac'].to_pandas()


In [22]:
data.to_hdf('geneCG_expr_corr/PBMC_NaiveMem_gene_mCG.hdf', key='data')


In [23]:
data = pd.read_hdf('geneCG_expr_corr/PBMC_NaiveMem_gene_mCG.hdf', key='data')


In [24]:
data.columns = data.columns.str.split('.').str[0]

In [25]:
expr = pd.read_hdf('PMD_ATAC/gene/bulkexpr_Hao2021.hdf', key='data')
design = pd.read_hdf('PMD_ATAC/gene/design_Hao2021.hdf', key='data')
expr_ct = expr.groupby(design['celltype']).sum()


In [26]:
selg = data.columns.intersection(expr_ct.columns)
print(len(selg))

In [27]:
data = data[selg]
expr_ct = expr_ct[selg]

In [28]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(3, 2, figsize=(8,9), dpi=300, sharex='all', sharey='all')
for i,ct in enumerate(data.index):
    ax = axes.flatten()[i]
    # print(pred.shape, label.shape, pearsonr(pred, label)[0])
    tmp1, tmp2 = data.loc[ct], expr_ct.loc[ct]
    tmp2 = np.log1p(tmp2 / tmp2.sum() * 1e6)
    hist = ax.hist2d(tmp1, tmp2, bins=100, cmap='Blues', 
                     # clim=[1e1, 1e4], 
                     norm=mpl.colors.LogNorm(),
                     range=[[0,1], [0,6]]
                    )
    ax.set_xlabel('mCG', fontsize=12)
    ax.set_ylabel('Expr', fontsize=12)
    cabr = plt.colorbar(hist[3], label='# genes', ax=ax)
    ax.set_title(f'{ct} Corr={np.around(pearsonr(tmp1, tmp2)[0], decimals=3)}')
    ax.set_ylim([0, 6])

plt.tight_layout()
# plt.savefig(f'HBAENTEx_mCH_corr_hist2d.pdf', transparent=True)


In [29]:
gene_meta = pd.read_csv(f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz', sep='\t', header=0)
gene_meta['gene_id_idx'] = gene_meta['gene_id'].str.split('.').str[0]
selg = gene_meta['gene_id_idx'].duplicated(keep=False)
gene_meta.loc[selg, 'gene_id_idx'] = gene_meta.loc[selg, 'gene_id']
genebed = gene_meta.set_index('gene_id_idx').loc[expr.columns].sort_values(by=['chrom', 'start', 'end'])


In [30]:
genecg_bw = []
for ct in data.index:
    tmp = pd.read_csv(f'PMD_ATAC/gene/allgene.split50.slop250kb.5kb.{ct}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
    ratio = tmp[5].values.reshape((-1, 150))
    cov = tmp[2].values.reshape((-1, 150))
    # print((cov==0).sum() / cov.shape[0] / cov.shape[1])
    ratio[cov==0] = np.nan
    genecg_bw.append(np.nanmean(ratio[:, 50:100], axis=1))
    print(ct)
    
genetmp = genebed.reset_index().loc[tmp.index.str.split('_').str[0].unique().astype(int), ['index', 'strand']].set_index('index')
genecg_bw = pd.DataFrame(genecg_bw, index=data.index, columns=genetmp.index)


In [31]:
selg = genecg_bw.columns.intersection(data.columns)
print(len(selg))


In [32]:
genecg_bw = genecg_bw[selg]
expr_ct = expr_ct[selg]
data = data[selg]

In [33]:
genecg_bw[0].shape

In [34]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(3, 2, figsize=(8,9), dpi=300, sharex='all', sharey='all')
for i,ct in enumerate(data.index):
    ax = axes.flatten()[i]
    # print(pred.shape, label.shape, pearsonr(pred, label)[0])
    tmp1, tmp2 = genecg_bw.loc[ct], expr_ct.loc[ct]
    tmp2 = np.log1p(tmp2 / tmp2.sum() * 1e6)
    hist = ax.hist2d(tmp1, tmp2, bins=100, cmap='Blues', 
                     # clim=[1e1, 1e4], 
                     norm=mpl.colors.LogNorm(),
                     range=[[0,1], [0,6]]
                    )
    ax.set_xlabel('mCG', fontsize=12)
    ax.set_ylabel('Expr', fontsize=12)
    cabr = plt.colorbar(hist[3], label='# genes', ax=ax)
    ax.set_title(f'{ct} Corr={np.around(pearsonr(tmp1, tmp2)[0], decimals=3)}')
    ax.set_ylim([0, 6])

plt.tight_layout()
# plt.savefig(f'HBAENTEx_mCH_corr_hist2d.pdf', transparent=True)


In [35]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(3, 2, figsize=(8,9), dpi=300, sharex='all', sharey='all')
for i,ct in enumerate(data.index):
    ax = axes.flatten()[i]
    # print(pred.shape, label.shape, pearsonr(pred, label)[0])
    tmp1, tmp2 = genecg_bw.loc[ct], data.loc[ct]
    hist = ax.hist2d(tmp1, tmp2, bins=100, cmap='Blues', 
                     # clim=[1e1, 1e4], 
                     norm=mpl.colors.LogNorm(),
                     range=[[0,1], [0,1]]
                    )
    ax.set_xlabel('mCG bw', fontsize=12)
    ax.set_ylabel('mCG allc', fontsize=12)
    cabr = plt.colorbar(hist[3], label='# genes', ax=ax)
    ax.set_title(f'{ct} Corr={np.around(pearsonr(tmp1, tmp2)[0], decimals=3)}')

plt.tight_layout()
# plt.savefig(f'HBAENTEx_mCH_corr_hist2d.pdf', transparent=True)
